In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set up paths
import os
from pathlib import Path

# Define base paths
DRIVE_PATH = "/content/drive/MyDrive"
PROJECT_PATH = os.path.join(DRIVE_PATH, "problem_statement_5")  # Adjust this to your actual folder name
TRAINING_PATH = os.path.join(PROJECT_PATH, "training", "train_PS05", "train")

# Create paths if they don't exist
os.makedirs(PROJECT_PATH, exist_ok=True)

# List contents to verify structure
print("=== Google Drive Structure ===")
print(f"Drive path: {DRIVE_PATH}")
print(f"Project path: {PROJECT_PATH}")
print(f"Training path: {TRAINING_PATH}")

# Check if training folder exists
if os.path.exists(TRAINING_PATH):
    print(f"\n✅ Training folder found!")
    # Count files
    json_files = len([f for f in os.listdir(TRAINING_PATH) if f.endswith('.json')])
    png_files = len([f for f in os.listdir(TRAINING_PATH) if f.endswith('.png')])
    print(f"📁 JSON files: {json_files}")
    print(f"🖼️ PNG files: {png_files}")

    # Show first few files
    files = os.listdir(TRAINING_PATH)[:10]
    print(f"\n📋 Sample files:")
    for f in files:
        print(f"  - {f}")
else:
    print(f"\n❌ Training folder not found at: {TRAINING_PATH}")
    print("Please check your folder structure and update PROJECT_PATH if needed")

# Set working directory
os.chdir(PROJECT_PATH)
print(f"\n📂 Working directory set to: {os.getcwd()}")

Mounted at /content/drive
=== Google Drive Structure ===
Drive path: /content/drive/MyDrive
Project path: /content/drive/MyDrive/problem_statement_5
Training path: /content/drive/MyDrive/problem_statement_5/training/train_PS05/train

✅ Training folder found!
📁 JSON files: 4008
🖼️ PNG files: 4010

📋 Sample files:
  - doc_04452.png
  - doc_04514.json
  - doc_04425.png
  - doc_04571.png
  - doc_04450.png
  - doc_04509.png
  - doc_04498.json
  - doc_04434.json
  - doc_04563.json
  - doc_04497.json

📂 Working directory set to: /content/drive/MyDrive/problem_statement_5


🔹 Goal

Take RT-DETR predictions (bbox, category) + original image → crop each detected region → store as base64 string + metadata (doc_id, region_id, category, bbox, confidence).

This becomes the “bridge format” to send into GPT-4o / Claude VLM for OCR, table extraction, figure captioning, etc.

In [ ]:
import os, json, base64
from PIL import Image
import io

# Input/Output paths
VAL_IMG_DIR = "/content/drive/MyDrive/problem_statement_5/datasets/training_ready/val/images"
VAL_PRED_DIR = "/content/drive/MyDrive/problem_statement_5/predictions/val"
CROPS_OUT_DIR = "/content/drive/MyDrive/problem_statement_5/vlm_ready"
os.makedirs(CROPS_OUT_DIR, exist_ok=True)

# Class map
id2name = {0:"text", 1:"title", 2:"list", 3:"table", 4:"figure"}

def encode_image(image: Image.Image) -> str:
    """Convert PIL image → base64 string"""
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

for pred_file in os.listdir(VAL_PRED_DIR):
    if not pred_file.endswith(".json"):
        continue
    doc_id = pred_file.replace(".json", "")
    img_path = os.path.join(VAL_IMG_DIR, doc_id + ".png")
    pred_path = os.path.join(VAL_PRED_DIR, pred_file)

    if not os.path.exists(img_path):
        continue

    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    # Load predictions
    with open(pred_path) as f:
        preds = json.load(f).get("annotations", [])

    region_entries = []
    for i, p in enumerate(preds):
        cls = int(p["category_id"])
        x, y, w, h = p["bbox"]
        x1, y1, x2, y2 = x, y, x + w, y + h

        # Crop region
        crop = img.crop((x1, y1, x2, y2))
        crop_b64 = encode_image(crop)

        # Build metadata entry
        entry = {
            "doc_id": doc_id,
            "region_id": i,
            "category": id2name.get(cls, str(cls)),
            "bbox": [x1, y1, x2, y2],
            "confidence": round(p.get("score", 0.0), 4),
            "image_base64": crop_b64
        }
        region_entries.append(entry)

    # Save one JSON file per document with all regions
    out_path = os.path.join(CROPS_OUT_DIR, f"{doc_id}_regions.json")
    with open(out_path, "w") as f:
        json.dump(region_entries, f, indent=2)

print(f"✅ Processed {len(os.listdir(CROPS_OUT_DIR))} region files into {CROPS_OUT_DIR}")


✅ Processed 4802 region files into /content/drive/MyDrive/problem_statement_5/vlm_ready


In [ ]:
import os
from google.colab import userdata  # ✅ works if you set secrets in Colab

# Pull API key securely
os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_AI")

from openai import OpenAI
client = OpenAI()   # ✅ picks up key from env var automatically


In [ ]:
#!pip install weave

In [ ]:
import os, json, wandb, weave
from glob import glob
from PIL import Image
import base64
from io import BytesIO

# === INIT PROJECT ===
wandb.init(project="rtdetr-checkpoints-showcase", job_type="vlm-extraction-sample")
weave.init("rtdetr-checkpoints-showcase")

# === PATHS ===
VLM_READY_DIR = "/content/drive/MyDrive/problem_statement_5/vlm_ready"
VLM_OUTPUT_DIR = "/content/drive/MyDrive/problem_statement_5/vlm_outputs"
os.makedirs(VLM_OUTPUT_DIR, exist_ok=True)

# === CLASS MAP + PROMPTS ===
id2name = {0:"text", 1:"title", 2:"list", 3:"table", 4:"figure"}

prompts = {
    "text": "You are an OCR assistant. Read the text in the image region. Output: {\"text\": \"...\"}",
    "title": "Extract the main title. Output: {\"title\": \"...\"}",
    "list": "Extract all list items. Output: {\"items\": [\"item1\", \"item2\"]}",
    "table": "Convert the table to JSON rows. Output: {\"table\": [{\"col1\": \"...\"}, ...]}",
    "figure": "Describe the figure type + caption. Output: {\"caption\": \"...\", \"figure_type\": \"...\"}"
}

# === OPENAI CLIENT ===
from openai import OpenAI
client = OpenAI()

# === WEAVE-TRACEABLE FUNCTION ===
@weave.op()
def run_vlm_on_region(region_dict):
    category = region_dict["category"]
    prompt = prompts[category]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": prompt + " (Return JSON strictly.)"},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": f"Category: {category}. Extract structured output in JSON."},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{region_dict['image_base64']}"}},
                ],
            },
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

# === W&B Table for inspection ===
vlm_table = wandb.Table(columns=["doc_id", "category", "region_image", "vlm_output"])

# === PROCESS 50 SAMPLES ===
results = []
region_files = glob(os.path.join(VLM_READY_DIR, "*.json"))
print(f"📂 Found {len(region_files)} region files")

for rf in region_files[:50]:   # limit for testing
    with open(rf) as f:
        regions = json.load(f)

    # If single dict, wrap in list
    if isinstance(regions, dict):
        regions = [regions]

    for region in regions:   # loop over each region dict
        try:
            output = run_vlm_on_region(region)
        except Exception as e:
            output = {"error": str(e)}

        result = {
            "doc_id": region["doc_id"],
            "region_id": region["region_id"],
            "category": region["category"],
            "bbox": region["bbox"],
            "vlm_output": output
        }

        # Save locally
        out_file = os.path.join(
            VLM_OUTPUT_DIR,
            f"{region['doc_id']}_{region['region_id']}.json"
        )
        with open(out_file, "w") as f:
            json.dump(result, f, indent=2)

        # Decode image for logging
        img_data = base64.b64decode(region["image_base64"])
        img = Image.open(BytesIO(img_data))

        # Add row to W&B table
        vlm_table.add_data(
            result["doc_id"],
            result["category"],
            wandb.Image(img, caption=f"{result['doc_id']} - {result['category']}"),
            json.dumps(output, indent=2)
        )

        results.append(result)

print(f"✅ Processed {len(results)} regions → {VLM_OUTPUT_DIR}")

# === Log sample outputs table ===
wandb.log({"vlm_sample_outputs": vlm_table})
print("✅ Logged 50-sample VLM outputs table to W&B")

wandb: Initializing weave.


Output()

weave: wandb version 0.22.1 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: Logged in as Weights & Biases user: rishabh29288.
weave: View Weave data at https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/weave
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ad-f9c4-7b09-b394-2b87c04e9fe7


📂 Found 4802 region files


weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-14ef-7eb3-9666-b11638e5e797
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-24d4-709d-ae41-82cd70f5727d
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-36cb-72cf-a6a1-dbccc195c308
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-455c-7113-a1b9-88861634e166
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-bca4-7f5f-8da7-286f381ac4ef
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-cf76-7a84-b907-939c0ccd6ff5
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-e7a4-7e91-9650-de2d6a7be0ae
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-ed20-7705-939d-432fdce8c94d
weave: 🍩 https://wandb.ai/rishabh29288/rtdetr-checkpoints-showcase/r/call/0199a6ae-f221-70f6-a163-5672e6afb8e6
w

✅ Processed 517 regions → /content/drive/MyDrive/problem_statement_5/vlm_outputs
✅ Logged 50-sample VLM outputs table to W&B


In [ ]:
import os, json, weave
from glob import glob
from openai import OpenAI

# === INIT ===
weave.init("rtdetr-checkpoints-showcase")
client = OpenAI()

# === PATHS ===
VLM_READY_DIR = "/content/drive/MyDrive/problem_statement_5/vlm_ready"
region_files = glob(os.path.join(VLM_READY_DIR, "*.json"))

# === MODEL OP ===
@weave.op()
def run_vlm_on_region(region_dict):
    """Run GPT-4o on one cropped region dict and normalize outputs"""
    category = region_dict["category"]
    prompt = f"Extract structured info from this {category} region. Respond only with JSON."

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": f"Process category: {category}"},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{region_dict['image_base64']}"},
                        },
                    ],
                },
            ],
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content

        # === Safe JSON parse + normalization ===
        parsed = json.loads(content) if content else {}
        safe_parsed = {k: str(v) for k, v in parsed.items()}  # normalize all values to strings
        return safe_parsed

    except Exception as e:
        return {"error": str(e)}

# === SCORERS ===
@weave.op()
def check_valid_json(model_output):
    """Programmatic scorer: ensure output is valid JSON with at least 1 key"""
    if not isinstance(model_output, dict):
        return {"valid": False, "reason": "Not a dict"}
    if len(model_output.keys()) == 0:
        return {"valid": False, "reason": "Empty JSON"}
    if "error" in model_output:
        return {"valid": False, "reason": model_output["error"]}
    return {"valid": True, "reason": ""}

eval_prompt = """
You are an OCR validation system. Validate whether the structured JSON matches the image content.
Return strictly valid JSON:
{"Correct": true, "Reason": null}
OR
{"Correct": false, "Reason": "short explanation"}
"""

@weave.op()
def llm_judge(model_output, image_base64):
    """LLM-as-a-judge scorer"""
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": eval_prompt},
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}},
                        {"type": "text", "text": str(model_output)},
                    ],
                },
            ],
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content
        return json.loads(content) if content else {"Correct": False, "Reason": "Empty response"}
    except Exception as e:
        return {"Correct": False, "Reason": f"LLM Judge Error: {str(e)}"}

# === BUILD DATASET (explicitly expose image_base64) ===
rows = []
for rf in region_files[:517]:   # all or limit
    with open(rf) as f:
        region = json.load(f)
    if isinstance(region, list):
        for r in region:
            rows.append({"region_dict": r, "image_base64": r["image_base64"]})
    else:
        rows.append({"region_dict": region, "image_base64": region["image_base64"]})

print(f"✅ Loaded {len(rows)} region dicts (with image_base64)")
dataset = weave.Dataset(rows=rows, name="vlm_regions_val")

# === EVALUATION with preprocess_model_input ===
def preprocess_model_input(row):
    # row is {"region_dict": {...}, "image_base64": "..."}
    return {"region_dict": row["region_dict"]}

evaluation = weave.Evaluation(
    dataset=dataset,
    scorers=[check_valid_json, llm_judge],
    name="StageB_VLM_eval",
    preprocess_model_input=preprocess_model_input,
)

print("🚀 Launching evaluation…")
result = await evaluation.evaluate(run_vlm_on_region)

# === PRETTY SUMMARY ===
def safe_summary(d):
    if isinstance(d, dict):
        clean = {}
        for k, v in d.items():
            if isinstance(v, (int, float)):
                clean[k] = v
            elif isinstance(v, dict):
                clean[k] = safe_summary(v)
            else:
                clean[k] = str(v)
        return clean
    return d

print("\n📊 Evaluation Summary:")
print(json.dumps(safe_summary(result), indent=2))

weave: retry_attempt


✅ Loaded 5451 region dicts (with image_base64)
🚀 Launching evaluation…


Streaming output truncated to the last 5000 lines.
weave: Evaluated 529 of 5451 examples
weave: Evaluated 530 of 5451 examples
weave: Evaluated 531 of 5451 examples
weave: Evaluated 532 of 5451 examples
weave: Evaluated 533 of 5451 examples
weave: Evaluated 534 of 5451 examples
weave: Evaluated 535 of 5451 examples
weave: Evaluated 536 of 5451 examples
weave: Evaluated 537 of 5451 examples
weave: Evaluated 538 of 5451 examples
weave: Evaluated 539 of 5451 examples
weave: Evaluated 540 of 5451 examples
weave: Evaluated 541 of 5451 examples
weave: Evaluated 542 of 5451 examples
weave: Evaluated 543 of 5451 examples
weave: Evaluated 544 of 5451 examples
weave: Evaluated 545 of 5451 examples
weave: Evaluated 546 of 5451 examples
weave: Evaluated 547 of 5451 examples
weave: Evaluated 548 of 5451 examples
weave: Evaluated 549 of 5451 examples
weave: Evaluated 550 of 5451 examples
weave: Evaluated 551 of 5451 examples
weave: Evaluated 552 of 5451 examples
weave: Evaluated 553 of 5451 examples


📊 Evaluation Summary:
{
  "check_valid_json": {
    "valid": {
      "true_count": 4929,
      "true_fraction": 0.9042377545404513
    }
  },
  "llm_judge": {
    "Correct": {
      "true_count": 3105,
      "true_fraction": 0.569620253164557
    }
  },
  "model_latency": {
    "mean": 12.198322343511377
  }
}


In [ ]:
import os, json, wandb, weave
from glob import glob
from PIL import Image
import base64
from io import BytesIO

# === INIT PROJECT ===
wandb.init(project="rtdetr-checkpoints-showcase", job_type="vlm-extraction-sample")
weave.init("rtdetr-checkpoints-showcase")

# === PATHS ===
VLM_READY_DIR = "/content/drive/MyDrive/problem_statement_5/vlm_ready"
VLM_OUTPUT_DIR = "/content/drive/MyDrive/problem_statement_5/vlm_outputs"
os.makedirs(VLM_OUTPUT_DIR, exist_ok=True)

# === CLASS MAP + PROMPTS ===
id2name = {0:"text", 1:"title", 2:"list", 3:"table", 4:"figure"}

prompts = {
    "text": "You are an OCR assistant. Read the text in the image region. Output: {\"text\": \"...\"}",
    "title": "Extract the main title. Output: {\"title\": \"...\"}",
    "list": "Extract all list items. Output: {\"items\": [\"item1\", \"item2\"]}",
    "table": "Convert the table to JSON rows. Output: {\"table\": [{\"col1\": \"...\"}, ...]}",
    "figure": "Describe the figure type + caption. Output: {\"caption\": \"...\", \"figure_type\": \"...\"}"
}

# === OPENAI CLIENT ===
from openai import OpenAI
client = OpenAI()

# === WEAVE-TRACEABLE FUNCTION ===
@weave.op()
def run_vlm_on_region(region_dict):
    category = region_dict["category"]
    prompt = prompts[category]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": prompt + " (Return JSON strictly.)"},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": f"Category: {category}. Extract structured output in JSON."},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{region_dict['image_base64']}"}},
                ],
            },
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

# === W&B Table for inspection ===
vlm_table = wandb.Table(columns=["doc_id", "category", "region_image", "vlm_output"])

# === PROCESS 50 SAMPLES ===
results = []
region_files = glob(os.path.join(VLM_READY_DIR, "*.json"))
print(f"📂 Found {len(region_files)} region files")

for rf in region_files[51:100]:   # limit for testing
    with open(rf) as f:
        regions = json.load(f)

    # If single dict, wrap in list
    if isinstance(regions, dict):
        regions = [regions]

    for region in regions:   # loop over each region dict
        try:
            output = run_vlm_on_region(region)
        except Exception as e:
            output = {"error": str(e)}

        result = {
            "doc_id": region["doc_id"],
            "region_id": region["region_id"],
            "category": region["category"],
            "bbox": region["bbox"],
            "vlm_output": output
        }

        # Save locally
        out_file = os.path.join(
            VLM_OUTPUT_DIR,
            f"{region['doc_id']}_{region['region_id']}.json"
        )
        with open(out_file, "w") as f:
            json.dump(result, f, indent=2)

        # Decode image for logging
        img_data = base64.b64decode(region["image_base64"])
        img = Image.open(BytesIO(img_data))

        # Add row to W&B table
        vlm_table.add_data(
            result["doc_id"],
            result["category"],
            wandb.Image(img, caption=f"{result['doc_id']} - {result['category']}"),
            json.dumps(output, indent=2)
        )

        results.append(result)

print(f"✅ Processed {len(results)} regions → {VLM_OUTPUT_DIR}")

# === Log sample outputs table ===
wandb.log({"vlm_sample_outputs": vlm_table})
print("✅ Logged 50-sample VLM outputs table to W&B")

⠴ Flushing tasks (63982 remaining: 63982 call-batch) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 11:56:29

In [ ]:
wandb.finish()